#We will fine-tune an ada classifier to distinguish between the two sorts : BaseBall and Hockey

In [1]:
!pip install --quiet openai

In [2]:
!pip install --upgrade openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.4 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.14.0
    Uninstalling openai-2.14.0:
      Successfully uninstalled openai-2.14.0


#Open ai api key pass

In [3]:
import os
import openai
import getpass
openai.api_key = getpass.getpass(prompt='OpenAI API Key:')

OpenAI API Key:··········


#Dataset import from scikit-learn

In [6]:
from sklearn.datasets import fetch_20newsgroups
import pandas as pd
import openai

categories = ['rec.sport.baseball', 'rec.sport.hockey']
sports_dataset = fetch_20newsgroups(subset='train',
                                      shuffle=True, random_state=42,
                                      categories=categories,
                                     )

#Data exploration
* The newsgroup dataset can be loaded using sklearn. first we will look at the data itself:

In [8]:
print(sports_dataset['data'][0])

From: dougb@comm.mot.com (Doug Bank)
Subject: Re: Info needed for Cleveland tickets
Reply-To: dougb@ecs.comm.mot.com
Organization: Motorola Land Mobile Products Sector
Distribution: usa
Nntp-Posting-Host: 145.1.146.35
Lines: 17

In article <1993Apr1.234031.4950@leland.Stanford.EDU>, bohnert@leland.Stanford.EDU (matthew bohnert) writes:

|> I'm going to be in Cleveland Thursday, April 15 to Sunday, April 18.
|> Does anybody know if the Tribe will be in town on those dates, and
|> if so, who're they playing and if tickets are available?

The tribe will be in town from April 16 to the 19th.
There are ALWAYS tickets available! (Though they are playing Toronto,
and many Toronto fans make the trip to Cleveland as it is easier to
get tickets in Cleveland than in Toronto.  Either way, I seriously
doubt they will sell out until the end of the season.)

-- 
Doug Bank                       Private Systems Division
dougb@ecs.comm.mot.com          Motorola Communications Sector
dougb@nwu.edu       

In [10]:
sports_dataset['target_names']

['rec.sport.baseball', 'rec.sport.hockey']

In [11]:
sports_dataset.target_names[sports_dataset['target'][0]]

'rec.sport.baseball'

In [12]:
sports_dataset.target_names[sports_dataset['target'][1]]

'rec.sport.hockey'

In [15]:
#len_baseball, len_hocky and all len
len_all, len_baseball, len_hocky = len(sports_dataset.data), len([e for e in sports_dataset.target if e == 0]), len([e for e in sports_dataset.target if e == 1])
print(f"Total examples: {len_all}, Baseball examples: {len_baseball}, Hockey examples: {len_hocky}")

Total examples: 1197, Baseball examples: 597, Hockey examples: 600


One sample from the baseball category can be seen above . it is an email to a mailing list. We can observe that we have 1197 examples in total, which are evenly split between the two sports.


#Data Preparation

We transform the dataset into a pandas dataframe, with a column for prompt and competion. The prompt contains the emaiil form the mailing list ,and the completion is a name of the sport , either hockey or baseball. For demonstration purpose only and speed of fine-tuing we take only 300 exdamples. in a real use case


In [18]:
from IPython.utils import text
import pandas as pd

labels = [sports_dataset.target_names[x].split('.')[-1] for x in sports_dataset['target']]
texts = [text.strip() for text in sports_dataset['data']]
df = pd.DataFrame(zip(texts, labels), columns=['prompt', 'completion'])
df.head()

,prompt,completion
0,From: dougb@comm.mot.com (Doug Bank)\nSubject:...,baseball
1,From: gld@cunixb.cc.columbia.edu (Gary L Dare)...,hockey
2,From: rudy@netcom.com (Rudy Wade)\nSubject: Re...,baseball
3,From: monack@helium.gas.uug.arizona.edu (david...,hockey
4,Subject: Let it be Known\nFrom: <ISSBTL@BYUVM....,baseball


** Both baseball and hockey are single tokens. We save the dataset as a jsonl file

In [19]:
df.to_json("sport.jsonl", orient='records', lines=True)


#Data Prepartion tool

We can now use a data prepartion tool which will suggest a few improvments to our dataset before fine-tuning. Before launching the tool we update the openai library to ensure we're using the latest data prepartion tool. We additionally specify -q which auto-accepts all suggestions.

In [20]:
!openai tools fine_tunes.prepare_data -f sports.jsonl -q


Analyzing...


ERROR in read_any_format validator: File sports.jsonl does not exist.

Aborting...

#Fine - tuning

The tool suggests we run the following command to train the dataset. Since this is a classificaton task, we would like to know what the generalization performance on the provided validation set is for our classification use case. The tool suggests to add --compute_classification_metrics --classification_positive_class "baseball" in order to comute the classification metrics.

In [21]:
ls


sample_data/  sport.jsonl


In [22]:
!openai --api-key "sk-dhglk-dkdgjlgj48ijggflghfgklgfgfkdj" api fine_tunes.create -t "sport_prepared_train.jsonl" -v "sport_prepared_valid.jsonl" --compute_classification_metrics --classification_positive_class "baseball" -m ada

usage: openai api [-h]
                  {chat.completions.create,images.generate,images.edit,images.create_variation,audio.transcriptions.create,audio.translations.create,files.create,files.retrieve,files.delete,files.list,models.list,models.retrieve,models.delete,completions.create,fine_tuning.jobs.create,fine_tuning.jobs.retrieve,fine_tuning.jobs.list,fine_tuning.jobs.cancel,fine_tuning.jobs.list_events}
                  ...
openai api: error: argument {chat.completions.create,images.generate,images.edit,images.create_variation,audio.transcriptions.create,audio.translations.create,files.create,files.retrieve,files.delete,files.list,models.list,models.retrieve,models.delete,completions.create,fine_tuning.jobs.create,fine_tuning.jobs.retrieve,fine_tuning.jobs.list,fine_tuning.jobs.cancel,fine_tuning.jobs.list_events}: invalid choice: 'fine_tunes.create' (choose from chat.completions.create, images.generate, images.edit, images.create_variation, audio.transcriptions.create, audio.tran

The accuracy reaches 99.6% On the plot below we can see how accuracy on the validation set increases during the training run.

In [23]:
results[results['classification/accuracy'].notnull()]['classification/accuracy'].plot()


#Using the model
We can now call the model to get the predictions.

In [24]:
test = pd.read_json('sport2_prepared_valid.json', lines=True)
test.head()

we need to use the same separtor following the prompt which we used during fine-tuning. in this case it is \n\n###\n\n. Since we are concerned with classification, we want the temperature to be as low as possible , and we only require one token completion to determine the prediction of the model.


In [ ]:
ft_model = 'ada:ft-openai-2021-07-30-12-26-20'
res = openai.Completion.create(model=ft_model,
                               prompt=test['prompt'][0] + '\n\n###\n\n',
                               max_token=1,
                               temperature=0

                               )
res['choices'][0]['text']



To get the long probabilities, we can specify logprobs parameter on the completion request

In [ ]:
res = openai.Completion.create(model=ft_model,
                               prompt=test['prompt'][0] + '\n\n###\n\n',
                               max_token=1,
                               temperature=0

                               )

res['choices'][0]['logprobs']['top_logprobs'][10]

We can see that the model predicts hockey as a lot more likely  then baseball , which is the correct prediction.
By requesting log_probs, we can see the prediction(log)
probability for each class.

#Generalization

initerestingly, our fine-tuned classifier is quit versatile , Despite being trained on emails to different mailing lists , is also succefully predicts tweets

In [ ]:
sample_hockey_tweet = """
Thank you to the
@Canes
and all you amazing Caniacs that have been so suportive! You guys are some of the best
fans in the NHL
@DetroitRedWings
"""

res = openai.Completion.create(model=ft_model,
                               prompt=sample_hockey_tweet + '\n\n###\n\n',
                               max_token=1,
                               temperature=0

                               )

res['choices'][0]['text']

#customo datar opure openai model ke fine tuning korte pari (native)